In [4]:
#!/usr/bin/env python3
import sys
import os
import csv
# using time module
import time

# ts stores the time in seconds
sts = time.time()

os.environ["XILINX_XRT"] = "/usr"
sys.path.append('/usr/local/share/pynq-venv/lib/python3.10/site-packages')

from pynq import Overlay
from pynq import allocate

def read_and_save_words(overlay):
    
    ###############################################################################################

    # Configure and trigger the hardware IP
    ctr_run_time = 100 # time period of run in number of 100MHz cycles

    ro_sel = 1 # select ring oscillator type: 1=not 3=nor 5&7=nand 

    sample_size = 100 # the number of samples saved by the script *has to be same as HW*
    ###############################################################################################  
    
    output_buffer = allocate(shape=(sample_size,), dtype='u4')
    output_buffer[:] = 0
    
    # Start the DMA receive channel
    overlay.axi_dma.recvchannel.start()
    
    overlay.rofreqIP.write(0x4, ctr_run_time) # send time period of run
    overlay.rofreqIP.write(0x0, ro_sel)  # Start the operation 
    overlay.rofreqIP.write(0x0, 0)  # Immediately stop the operation

    # Transfer data from the hardware to the buffer
    overlay.axi_dma.recvchannel.transfer(output_buffer)
    overlay.axi_dma.recvchannel.stop()

    # Save the output buffer to a CSV file
    with open('output_data_not.csv', 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)

        # Write each word in a single column
        for word in output_buffer:
           # print(f"Writing word: {word}")
            writer.writerow([word])

            
    
def main():

    # Load the overlay
    overlay = Overlay("rofc.bit")

    # Call the function to read and save words
    read_and_save_words(overlay)


    # ts stores the time in seconds
    fts = time.time()

    # print the current timestamp
    print(f"Completed run in: {(fts - sts)/60} minutes")
    

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"Program terminated due to error: {str(e)}")

Completed run in: 0.01742548147837321 minutes
